# 04 — Analysis

Distributions of schizotypal traits, cannabis use correlation, statistical tests.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")

SEED_POST_ID = "REPLACE_ME"

In [ ]:
scored_df = pl.read_parquet(f"data/processed/scored_{SEED_POST_ID}.parquet")

TRAIT_COLS = ["magical_thinking", "ideas_of_reference", "unusual_perceptions",
             "paranoid_ideation", "odd_speech", "social_anxiety"]

scored_df = scored_df.with_columns(
    pl.sum_horizontal(TRAIT_COLS).alias("total_score")
)

print(f"{len(scored_df)} scored posts from {scored_df['username'].n_unique()} users")

## Post-level distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, TRAIT_COLS):
    vals = scored_df[col].to_list()
    ax.hist(vals, bins=range(7), align="left", edgecolor="black")
    ax.set_title(col.replace("_", " ").title())
    ax.set_xlabel("Score")
    ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## User-level aggregation

Multiple aggregation strategies — not just mean, since most posts may be mundane.

In [ ]:
SCHIZO_THRESHOLD = 3  # posts scoring >= this on any trait count as "schizotypal"

user_agg = scored_df.group_by("username").agg(
    # Basic stats
    pl.col("total_score").mean().alias("mean_total"),
    pl.col("total_score").max().alias("max_total"),
    pl.col("total_score").quantile(0.9).alias("p90_total"),

    # Per-trait means
    *[pl.col(c).mean().alias(f"mean_{c}") for c in TRAIT_COLS],

    # Per-trait max
    *[pl.col(c).max().alias(f"max_{c}") for c in TRAIT_COLS],

    # Proportion of posts above threshold (on any trait)
    (pl.max_horizontal(TRAIT_COLS) >= SCHIZO_THRESHOLD).mean().alias("prop_schizotypal_posts"),

    # Cannabis
    pl.col("cannabis_mention").any().alias("any_cannabis_mention"),
    pl.col("cannabis_mention").sum().alias("cannabis_post_count"),

    # Volume
    pl.len().alias("post_count"),
)

user_agg.head(10)

## Cannabis users vs. non-users

In [ ]:
cannabis_users = user_agg.filter(pl.col("any_cannabis_mention"))
non_cannabis = user_agg.filter(~pl.col("any_cannabis_mention"))

print(f"Cannabis-mentioning users: {len(cannabis_users)}")
print(f"Non-cannabis users: {len(non_cannabis)}")

In [ ]:
# Compare distributions with box/violin plots
compare_cols = ["mean_total", "max_total", "p90_total", "prop_schizotypal_posts"]

fig, axes = plt.subplots(1, len(compare_cols), figsize=(16, 5))
for ax, col in zip(axes, compare_cols):
    data = [
        cannabis_users[col].to_list(),
        non_cannabis[col].to_list(),
    ]
    ax.boxplot(data, labels=["Cannabis", "No cannabis"])
    ax.set_title(col.replace("_", " ").title())
plt.tight_layout()
plt.show()

In [ ]:
# Statistical tests
for col in compare_cols:
    a = cannabis_users[col].drop_nulls().to_list()
    b = non_cannabis[col].drop_nulls().to_list()
    if len(a) < 2 or len(b) < 2:
        print(f"{col}: not enough data for test")
        continue
    u_stat, p_val = stats.mannwhitneyu(a, b, alternative="two-sided")
    # Effect size: rank-biserial correlation
    r = 1 - (2 * u_stat) / (len(a) * len(b))
    print(f"{col}: U={u_stat:.0f}, p={p_val:.4f}, r={r:.3f} (n={len(a)} vs {len(b)})")

## Per-trait breakdown by cannabis use

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, trait in zip(axes.flat, TRAIT_COLS):
    mean_col = f"mean_{trait}"
    a = cannabis_users[mean_col].drop_nulls().to_list()
    b = non_cannabis[mean_col].drop_nulls().to_list()
    ax.boxplot([a, b], labels=["Cannabis", "No cannabis"])
    ax.set_title(trait.replace("_", " ").title())
plt.suptitle("Mean trait scores: cannabis users vs. non-users", y=1.02)
plt.tight_layout()
plt.show()